# Data Collection

## NYT

In [6]:
import os, requests, pandas as pd, time, datetime as dt, random
from dotenv import load_dotenv
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

load_dotenv()
API_KEY = os.getenv("NYT_API_KEY")
if not API_KEY:
    raise RuntimeError("Set NYT_API_KEY in your .env file")

BASE = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

PHRASES = [
    "healthy aging", 
    # "healthy ageing", -> this query brings no articles at all
    "anti-aging", "anti-ageing", "anti aging", "anti ageing",
    "biohacking", "living longer",
    "well aging", "well ageing", "aging well", "ageing well",
    "slow aging", "slow ageing",
    "health rejuvenation", "nootropics",
]

def build_fq(phrases, sections=None):
    quoted = " ".join(f'"{p}"' for p in phrases)
    fields = ["abstract", "headline", "lead_paragraph", "snippet"]
    term_block = " OR ".join(f'{f}:({quoted})' for f in fields)
    if sections:
        sect_block = "section_name:(" + " ".join(f'"{s}"' for s in sections) + ")"
        return f"({term_block}) AND ({sect_block})"
    return f"({term_block})"

# ---- session with retries (handles TCP hiccups/5xx automatically) ----
session = requests.Session()
session.headers.update({"User-Agent": "nyt-healthy-aging/1.0"})
session.params = {"api-key": API_KEY}
retry = Retry(
    total=5,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    raise_on_status=False,
)
session.mount("https://", HTTPAdapter(max_retries=retry))

def get_json_with_backoff(params, max_retries=8):
    """
    Make a single NYT API call. If 429, honor Retry-After or use exponential backoff.
    """
    backoff = 8  # start gentle; NYT default is ~10 req/min, keep <9/min
    for attempt in range(1, max_retries + 1):
        r = session.get(BASE, params=params, timeout=60)
        if r.status_code == 429:
            ra = r.headers.get("Retry-After")
            sleep_s = int(ra) if (ra and ra.isdigit()) else backoff + random.uniform(0, 2)
            print(f"429: sleeping {sleep_s:.1f}s (attempt {attempt}/{max_retries})")
            time.sleep(sleep_s)
            backoff = min(backoff * 1.6, 120)
            continue
        # for 5xx the Retry() adapter already retried; if still bad, wait and retry
        if 500 <= r.status_code < 600:
            print(f"{r.status_code}: sleeping {backoff:.1f}s (attempt {attempt}/{max_retries})")
            time.sleep(backoff + random.uniform(0, 2))
            backoff = min(backoff * 1.6, 120)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError("Gave up after repeated 429/5xx responses (likely daily cap).")

def fetch_year(year, fq):
    begin, end = f"{year}0101", f"{year}1231"
    page, docs = 0, []
    while True:
        params = {
            "fq": fq,
            "begin_date": begin,
            "end_date": end,
            "page": page,
            "sort": "newest",
            # optional to shrink payload: return only fields you need
            # "fl": "web_url,headline,abstract,snippet,lead_paragraph,pub_date,section_name,_id"
        }
        j = get_json_with_backoff(params)
        if page == 0:
            meta = j.get("response", {}).get("meta", {})
            print(f"{year}: hits={meta.get('hits')}")
        batch = j["response"]["docs"]
        if not batch:
            break
        docs.extend(batch)
        page += 1
        if page >= 100:   # NYT hard cap per query
            break
        # stay well below the minute cap; add jitter so calls don't align
        time.sleep(8 + random.uniform(0, 2))
    return docs

def collect_nyt_articles(start_year=2010, end_year=None):
    if end_year is None:
        end_year = dt.datetime.today().year
    fq = build_fq(PHRASES, sections=None)
    all_docs = []
    for y in tqdm(range(start_year, end_year + 1), desc="NYT years"):
        all_docs.extend(fetch_year(y, fq))
    df = pd.json_normalize(all_docs)
    if not df.empty and "_id" in df.columns:
        df = df.drop_duplicates("_id")
    return df

if __name__ == "__main__":
    df = collect_nyt_articles(start_year=2020, end_year=2021)
    out = "../NYT/Data_NYT/nyt_healthy_ageing_2010_present.csv"
    df.to_csv(out, index=False)
    print(f"✔ Saved {len(df):,} articles to {out}")


NYT years:  50%|█████████████████                 | 1/2 [00:01<00:01,  1.75s/it]

2020: hits=None


NYT years: 100%|██████████████████████████████████| 2/2 [00:02<00:00,  1.24s/it]

2021: hits=None
✔ Saved 0 articles to ../NYT/Data_NYT/nyt_healthy_ageing_2010_present.csv


## Guardian API

Manual for using API:
https://open-platform.theguardian.com/documentation/

### Searching Functions

In [1]:
import os, time,random, datetime as dt, requests, pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from dotenv import load_dotenv                       

load_dotenv() 
API_KEY = os.getenv("GUARDIAN_API_KEY") 
BASE_URL = "https://content.guardianapis.com/search"

# constant values
FIELDS = "bodyText,headline,trailText,byline,wordcount"
TAGS   = "keyword,contributor"
BLOCKS = "body"  
KEYWORDS=[
    "aging","ageing",
    "healthy aging","healthy ageing",
    "longevity",
    "anti-ageing","anti-aging","anti ageing","anti aging",
    "biohacking", "living longer", "well ageing", "well aging", 
    "ageing well", "aging well",
    "slow ageing", "slow aging",
    "health rejuvenation", "nootropics"
]

PAGE_SIZE = 200       
SLEEP_SEC = 1.2

In [2]:
def build_query(key, core_keys, force_and=False, force_OR = False):
    key = [f'"{k}"' for k in key]
    if force_OR:
        return " OR ".join(key)
    # ("cell" OR "cells") AND (<core...>)  or just ("midlife" OR "midlives")
    key = key.strip('"').strip()
    variants = [f'{key}']
    if not key.endswith('s'): variants.append(f'{key}s')
    lhs = "(" + " OR ".join(dict.fromkeys(variants)) + ")"
    if not force_and:
        return lhs
    rhs = "(" + " OR ".join([f'"{k}"' if " " in k else k for k in core_keys]) + ")"
    return f"{lhs} AND {rhs}"

def fetch_page(page: int, *, query: str, from_date: str, to_date: str,
               page_size: int = PAGE_SIZE,
               query_fields: str | None = None,
               order_by: str | None = None,
               section: str | None = None,
               tag: str | None = None,
               content_type: str | None = None,
                max_retries: int = 5,
               backoff_base: float = 1.5) -> dict:
    params = {
        "from-date": from_date,
        "to-date": to_date,
        "page-size": page_size,
        "page": page,
        "show-fields": FIELDS,
        "show-tags": TAGS,
        "show-blocks": BLOCKS,
        "use-date": "published",
        "api-key": API_KEY,
    }
    if query_fields:
        params["query-fields"] = query_fields       # <— important
    if order_by:
        params["order-by"] = order_by
    if section:
        params["section"] = section
    if content_type:
        params["type"] = content_type
    if tag:          
        params["tag"] = tag 
    if query:
        params["q"] = query
    
    attempt = 0
    while True:
        resp = None
        try:
            resp = requests.get(BASE_URL, params=params, timeout=60)
            resp.raise_for_status()
            return resp.json()["response"]
        except requests.HTTPError:
            status = resp.status_code if resp is not None else None
            # backoff on 429 + transient 5xx
            if status in (429, 500, 502, 503, 504) and attempt < max_retries:
                delay = (backoff_base ** attempt) + random.random()
                time.sleep(delay)
                attempt += 1
                continue
            raise
        except requests.RequestException:
            # network hiccup / timeout etc.
            if attempt < max_retries:
                delay = (backoff_base ** attempt) + random.random()
                time.sleep(delay)
                attempt += 1
                continue
            raise
    # debug_url = requests.Request("GET", BASE_URL, params=params).prepare().url
    # print("URL:", debug_url)

def extract_links(html: str) -> list[str]:
    """Return all href links from an HTML fragment."""
    soup = BeautifulSoup(html or "", "html.parser")
    return [a["href"] for a in soup.find_all("a", href=True)]

def run(KEYS: list[str],
        FROM: str,
        TO: str,
        CORE_KEYS:list[str] = ["aging","ageing","healthy aging","healthy ageing","longevity","anti-ageing","anti-aging","anti ageing","anti aging","biohacking", "health rejuvenation", "nootropics"],
        PATH: str | None = None,
        FILENAME: str | None = None,
        max_pages: int | None = None, #the maximum number of results pages to fetch from the api
        max_articles: int | None = None, # the maximum total number of articles we want
        page_size: int = PAGE_SIZE, #page_size = articles per page
        query_fields: str | None = None,
        order_by: str | None = None,
        section: str | None = None,
        tag: str | None = None,
        content_type: str | None = None,
        force_OR: bool = False,
        force_AND: bool = True):
     
    if not KEYS:                       # tag-only mode
        QUERY = None
    else:
        QUERY = build_query(KEYS, CORE_KEYS, force_AND, force_OR)
    print("QUERY:", QUERY)

    first = fetch_page(1, query=QUERY, from_date=FROM, to_date=TO,
                       page_size=page_size, query_fields=query_fields,
                       order_by=order_by, section=section, tag=tag,
                       content_type=content_type)
    total_results = first["total"]
    print(f"{KEYS}: Total matching articles found = {total_results:,}")
    total_pages = first["pages"]
    pages = min(total_pages, max_pages) if max_pages else total_pages
    results = list(first["results"])

    pbar = tqdm(range(2, pages + 1), desc="Guardian pages")
    for p in pbar:
        time.sleep(SLEEP_SEC)
        resp = fetch_page(p, query=QUERY, from_date=FROM, to_date=TO, page_size=page_size, query_fields = query_fields, order_by=order_by, section=section,tag=tag, content_type=content_type)
        results.extend(resp["results"])
        #in case the length of results exceeds the maximum number of articles needed -> cut it
        if max_articles and len(results) >= max_articles:
            results = results[:max_articles]
            break

    # Flatten into a DataFrame
    rows = []
    for item in results:
        fields = item.get("fields", {})
        blocks = item.get("blocks", {})
        
        body_text = fields.get("bodyText", "")
        body_html = fields.get("bodyHtml", "")
        
        body_blocks = blocks.get("body", [])   # list of block dicts
        
        # blocks fallback
        if not body_html and body_blocks:
            body_html = "\n".join(b.get("bodyHtml", "") for b in body_blocks)
        if not body_text and body_blocks:
            body_text = "\n\n".join(b.get("bodyTextSummary", "") for b in body_blocks)
        
        links = extract_links(body_html)

        # authors from contributor tags
        authors = [t["webTitle"] for t in item.get("tags", []) if t.get("type") == "contributor"]
        keywords = [t["webTitle"] for t in item.get("tags", []) if t.get("type") == "keyword"]

        rows.append({
            "id": item.get("id"),
            "section": item.get("sectionName"),
            "pub_date": item.get("webPublicationDate"),
            "web_url": item.get("webUrl"),
            "headline": fields.get("headline"),
            "trailText": fields.get("trailText"),
            "byline": fields.get("byline"),
            "wordcount": fields.get("wordcount"),
            "bodyText": body_text,
            "links": "; ".join(links),
            "authors": "; ".join(authors),
            "tags": "; ".join(keywords),
        })

    df = pd.DataFrame(rows)

    return df

### Search Topic-Wise data by Topic Keys + Core Keys

In [6]:
def search_topic_by_keys(keys, filename, from_date, to_date,max_pages=5, page_size=50, ageing_tag_in = True, per_key=True):
    PATH = "../GuardianAPI/"
    FILENAME = filename
    FROM= from_date
    TO = to_date

    all_topics = []
    
    core_keys = [
             # "ageing", "aging"
             "healthy aging","healthy ageing",
             # "longevity",
             "anti-ageing","anti-aging","anti ageing","anti aging",
             "biohacking", "living longer", "well ageing", "well aging", 
             "ageing well", "aging well",
             "slow ageing", "slow aging",
             "health rejuvenation", "nootropics"]

    if keys:
        for key in keys:
            df = run(key, FROM, TO, CORE_KEYS = core_keys, query_fields="headline,body,trailText", max_pages=max_pages, max_articles=None, 
                     page_size=page_size, order_by="relevance",tag="science/ageing", content_type="article", force_OR = False, force_AND=True)
            if not df.empty:
                df = df.copy()
                df["topic_key"] = key
    else:
        if (per_key):
            for key in core_keys:
                df = run([key], FROM, TO, query_fields="headline,body,trailText", max_pages=max_pages, max_articles=None, 
                         page_size=page_size, order_by="relevance",tag=("science/ageing" if ageing_tag_in else None), content_type="article", force_OR=False)
                if not df.empty:
                    df = df.copy()
                    df["topic_key"] = key
        else:
            df = run(core_keys, FROM, TO, query_fields="headline,body,trailText", max_pages=max_pages, max_articles=None, 
                     page_size=page_size, order_by="relevance",tag=("science/ageing" if ageing_tag_in else None), content_type="article", force_OR=True)
            
    
    all_topics.append(df)
        
    combined = pd.concat(all_topics, ignore_index=True, sort=False)
    combined.drop_duplicates(subset=["id"], inplace=True)
        
    os.makedirs(PATH, exist_ok=True)
    out_file = os.path.join(PATH, FILENAME)
    combined.to_csv(out_file, index=False)
    print(f"Saved {len(combined)} rows to {out_file}")

### Search Topic-Wise data by Topic keys + Tags

In [4]:
def search_topic_by_tag(topic_keys, filename, from_date, to_date):
    PATH = "../GuardianAPI/"
    FILENAME = filename
    FROM= from_date
    TO = to_date

    all_topics = []

    core_keys = ["aging","ageing",
         "healthy aging","healthy ageing",
         "longevity",
         "anti-ageing","anti-aging","anti ageing","anti aging",
         "biohacking", "living longer", "well ageing", "well aging", 
         "ageing well", "aging well",
         "slow ageing", "slow aging",
         "health rejuvenation", "nootropics"]
    
    for key in topic_keys:
        df = run(key, FROM, TO, CORE_KEYS = core_keys, query_fields="headline,body,trailText", max_pages=5, max_articles=None, 
                 page_size=100, order_by="relevance",tag="science/ageing", content_type="article", force_OR=False, force_AND = False)
        
        if not df.empty:
                df = df.copy()
                df["topic_key"] = key
        
        all_topics.append(df)
        
    combined = pd.concat(all_topics, ignore_index=True, sort=False)
    combined.drop_duplicates(subset=["id"], inplace=True)
        
    os.makedirs(PATH, exist_ok=True)
    out_file = os.path.join(PATH, FILENAME)
    combined.to_csv(out_file, index=False)
    print(f"Saved {len(combined)} rows to {out_file}")

### Normalize Topic Keywords

In [5]:
import re, ast, os, itertools
from pathlib import Path
import spacy
nlp = spacy.load("en_core_web_sm")

ALLOW_POS = {"NOUN", "ADJ", "VERB",  "PROPN", "SYM"}   # contentful words
MIN_LEN = 3                           # drop very short tokens

def _normalize_single_token(tok):
    if tok.is_punct or tok.is_space or tok.like_num:
        return None
    if tok.is_stop:
        return None
    if tok.ent_type_ in {"GPE"}:
        return None
    if tok.pos_ not in ALLOW_POS:
        return None
    lemma = tok.lemma_.lower().strip()
    if len(lemma) < MIN_LEN:
        return None
    return lemma

def normalize_keyword(kw: str):
    """
    If phrase -> normalize token-by-token, keep phrase if something survives.
    If single token -> return normalized lemma or None.
    """
    text = (kw or "").strip().strip("'\"")
    if not text:
        return None

    doc = nlp(text)
    if " " in text:
        kept = []
        for tok in doc:
            norm = _normalize_single_token(tok)
            if norm:
                kept.append(norm)
        return " ".join(kept) if kept else None
    else:
        return _normalize_single_token(doc[0])

def read_lists_from_file(path: str):
    """
    Reads a file that contains Python list literals (one per line, or wrapped).
    Returns: list[list[str]]
    Works by extracting every [...] block from the whole file text.
    """
    text = Path(path).read_text(encoding="utf-8")
    # Grab each list literal between [ and ], non-greedy across newlines
    raw_lists = re.findall(r"\[.*?\]", text, flags=re.S)
    groups = []
    for chunk in raw_lists:
        try:
            lst = ast.literal_eval(chunk)
            # Ensure all items are strings
            lst = [str(x) for x in lst]
            groups.append(lst)
        except Exception:
            # If a particular chunk is malformed, skip it safely
            continue
    return groups

def flatten(list_of_lists):
    return list(itertools.chain.from_iterable(list_of_lists))

# ---- main preprocessing ----

def preprocess_keyword_groups(
    current_path: str,
    prev_path: str | None = None,
    return_flattened: bool = False
):
    curr_groups = read_lists_from_file(current_path)

    prev_norm_set = set()
    if prev_path and os.path.exists(prev_path):
        prev_groups = read_lists_from_file(prev_path)
        for kw in flatten(prev_groups):
            nk = normalize_keyword(kw)
            prev_norm_set.add(nk if nk is not None else kw.strip().lower())

    audit = {
        "dropped_prev_dup": [],
        "dropped_this_round_dup": []
    }

    cleaned_groups = []
    seen_this_round = set()  # normalized (or fallback) keys across whole round

    for group in curr_groups:
        group_kept = []
        group_seen = set()   # normalized (or fallback) keys within this group

        for kw in group:
            nk = normalize_keyword(kw)
            norm_key = nk if nk is not None else kw.strip().lower()

            if norm_key in prev_norm_set:
                audit["dropped_prev_dup"].append(kw)
                continue
            if norm_key in group_seen or norm_key in seen_this_round:
                audit["dropped_this_round_dup"].append(kw)
                continue

            group_kept.append(kw)    # keep the original surface form
            group_seen.add(norm_key)
            seen_this_round.add(norm_key)

        if group_kept:
            cleaned_groups.append(group_kept)

    if return_flattened:
        return cleaned_groups, flatten(cleaned_groups), audit
    return cleaned_groups, audit

/home/iskender/anaconda3/envs/bertopic-env/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/iskender/anaconda3/envs/bertopic-env/lib/python3.10/site-packages/torch/cuda/__init__.py:905: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 9010). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647382455/work/c10/cuda/CUDAFunctions.cpp:108.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


In [14]:
# Example:
cleaned_groups, audit = preprocess_keyword_groups(
    current_path="round1_keys.txt",
    prev_path="round0_keys.txt",      # or None
    return_flattened=False            # keep list-of-lists
)

# If you also want a flattened list for quick inspection:
cleaned_groups, flat_list, audit = preprocess_keyword_groups(
    current_path="round1_keys.txt",
    prev_path="round0_keys.txt",
    return_flattened=True
)

print(f"Groups kept: {len(cleaned_groups)}; total terms kept: {len(flat_list)}")
for k, v in audit.items():
    print(k, len(v))
cleaned_groups

Groups kept: 15; total terms kept: 310
dropped_prev_dup 138
dropped_this_round_dup 32


[['drivers',
  'species',
  'cannabis',
  'psychedelics',
  'lifespan',
  'discrimination',
  'ageism',
  'johnson',
  'bladder',
  'suit',
  'marriage',
  'temperate',
  'nest',
  'inflammation',
  'lab',
  'dalman'],
 ['income',
  'employment',
  'wealth',
  'tax',
  'boomers',
  'economy',
  'calment',
  'retire',
  'zak',
  'housing',
  'teacher',
  'afford',
  'frank',
  'baby boomers',
  'stereotypes'],
 ['genetic',
  'extend',
  'live longer',
  'immortality',
  'epigenetic',
  'senescence',
  'reverse',
  'horvath',
  'therapies'],
 ['care homes',
  'commission',
  'royal commission',
  'cohousing',
  'nursing home',
  'australians',
  'facility',
  'covid 19',
  'care quality',
  'home care',
  'care sector',
  'nursery',
  'residential aged',
  'standards',
  'ratios',
  'residential care',
  'elderly people',
  'appearance',
  'models',
  'filler',
  'acid',
  'sunscreen',
  'makeup',
  'uv',
  'moisturiser',
  'brands'],
 ['june',
  'read guardian',
  'partridge',
  'today 

## NewsApi.ai

In [22]:
from eventregistry import EventRegistry, QueryArticlesIter
import pandas as pd

er = EventRegistry(apiKey="a8bb5eaf-4e7f-4454-b614-ec2bfab00978")

queries = [
    "healthy ageing",
    "healthy aging",
    "anti-aging",
    "anti-ageing",
    "anti aging",
    "anti ageing",
    "biohacking",
    "living longer",
    "well aging",
    "well ageing",
    "aging well",
    "ageing well"
    "slow aging",
    "slow ageing",
    "health rejuvenation",
    "nootropics"
]

rows = []

for kw in queries:
    q = QueryArticlesIter(
        keywords=kw,
        lang="eng",
        dateStart="2010-01-01",
        dateEnd="2025-08-22"
    )
    for art in q.execQuery(er, maxItems=1000):
        rows.append({
            "title": art.get("title"),
            "body": art.get("body"),
            "url": art.get("url"),
            "source": art.get("source", {}).get("title"),
            "pub_datetime": art.get("dateTimePub"),
            "lang": art.get("lang"),
        })

df = pd.DataFrame(rows).drop_duplicates("url")
print("Total articles:", len(df))

Total articles: 4300


In [25]:
pd.set_option('display.max_colwidth', 60)
df.to_csv("../newsapi.ai/core_keys_data.csv", index=False)

## VOA

In [7]:
import json, pandas as pd
from pathlib import Path

# Where the JSONs actually live
BASE = Path("~/Desktop/Projects/MOT_VOA/mot").expanduser()

hits = Path("~/Desktop/Projects/MOT_VOA/mot/searches_dir/longevity_hits.txt").expanduser()
rel_files = [Path(line.strip()) for line in hits.read_text().splitlines() if line.strip()]

# Make them absolute
files = [(BASE / p).resolve() for p in rel_files]

rows = []
missing = []
for fp in files:
    if fp.exists():
        rows.append(json.loads(fp.read_text()))
    else:
        missing.append(str(fp))

df = pd.json_normalize(rows)
print("Loaded:", len(df), "Missing:", len(missing))


Loaded: 35468 Missing: 0


In [9]:
df.to_csv("../VOA/longevity_data.csv", index=False)

# Base Fetching

In [25]:
search_data_by_keys("guardian_data_base.csv", "2010-01-01","2025-12-31",per_key=False)

QUERY: aging OR ageing OR healthy aging OR healthy ageing OR longevity OR anti-ageing OR anti-aging OR anti ageing OR anti aging OR biohacking OR living longer OR well ageing OR well aging OR ageing well OR aging well OR slow ageing OR slow aging OR health rejuvenation OR nootropics
['aging', 'ageing', 'healthy aging', 'healthy ageing', 'longevity', 'anti-ageing', 'anti-aging', 'anti ageing', 'anti aging', 'biohacking', 'living longer', 'well ageing', 'well aging', 'ageing well', 'aging well', 'slow ageing', 'slow aging', 'health rejuvenation', 'nootropics']: Total matching articles found = 906


Guardian pages: 100%|███████████████████████████| 18/18 [00:48<00:00,  2.70s/it]


Saved 906 rows to ../GuardianAPI/guardian_data_base.csv


In [40]:
search_data_by_keys("guardian_data_base_per_key.csv", "2010-01-01","2025-12-31",per_key=True)

QUERY: aging
['aging']: Total matching articles found = 116


Guardian pages: 100%|█████████████████████████████| 2/2 [00:06<00:00,  3.43s/it]


QUERY: ageing
['ageing']: Total matching articles found = 746


Guardian pages: 100%|███████████████████████████| 14/14 [00:54<00:00,  3.91s/it]


QUERY: healthy aging
['healthy aging']: Total matching articles found = 142


...

QUERY: health rejuvenation
['health rejuvenation']: Total matching articles found = 113


Guardian pages: 100%|█████████████████████████████| 2/2 [00:05<00:00,  2.82s/it]


QUERY: nootropics
['nootropics']: Total matching articles found = 2


Guardian pages: 0it [00:00, ?it/s]


Saved 6936 rows to ../GuardianAPI/guardian_data_base_per_key.csv


In [59]:
keys = ["cell", 'mice', 'lifespan', 'cell', 'telomeres', 'senescent',
            'stem', 'plasma', 'dna', 'senescent cells', 'genetic', 'age related',
            'gene', 'genes', 'biological age', 'scientific', 'trial', 'trials',
            'centenarians', 'stem cells', 'young blood', 'immune', 'species',
            'japan', 'proteins', 'tissues', 'wyss', 'coray', 'wyss coray', 'marmalade']

search_topic_by_tag(keys, "guardian_data_topic1_per_key.csv", "2010-01-01","2025-12-31")

QUERY: (cell OR cells)
cell: Total matching articles found = 153


Guardian pages: 100%|█████████████████████████████| 1/1 [00:02<00:00,  2.73s/it]


QUERY: (mice OR mices)
mice: Total matching articles found = 78


Guardian pages: 0it [00:00, ?it/s]

QUERY: (lifespan OR lifespans)


...

Guardian pages: 0it [00:00, ?it/s]

QUERY: (marmalade OR marmalades)


marmalade: Total matching articles found = 17


Guardian pages: 0it [00:00, ?it/s]

Saved 823 rows to ../GuardianAPI/guardian_data_topic1_per_key.csv


In [28]:
df = run(KEYS=None, FROM = "2010-01-01",TO = "2025-12-31", query_fields="headline,body,trailText", max_pages=40, max_articles=None, 
                 page_size=50, order_by="relevance",tag="science/ageing", content_type="article", force_OR=False, force_AND=False)

QUERY: None
None: Total matching articles found = 1,508


Guardian pages:   0%|                                    | 0/30 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [38]:
search_topic_by_keys(keys=None, filename="guardian_base_across_tags.csv", from_date="2010-01-01", to_date="2025-12-31",max_pages = 1, page_size=100, ageing_tag_in = False, per_key=False)

QUERY: healthy aging OR healthy ageing OR longevity OR anti-ageing OR anti-aging OR anti ageing OR anti aging OR biohacking OR living longer OR well ageing OR well aging OR ageing well OR aging well OR slow ageing OR slow aging OR health rejuvenation OR nootropics
['healthy aging', 'healthy ageing', 'longevity', 'anti-ageing', 'anti-aging', 'anti ageing', 'anti aging', 'biohacking', 'living longer', 'well ageing', 'well aging', 'ageing well', 'aging well', 'slow ageing', 'slow aging', 'health rejuvenation', 'nootropics']: Total matching articles found = 92,829


Guardian pages: 0it [00:00, ?it/s]

Saved 100 rows to ../GuardianAPI/guardian_base_across_tags.csv


In [46]:
g = pd.read_csv("../GuardianAPI/guardian_base_across_tags.csv")

In [7]:
search_topic_by_keys(keys=None, filename="guardian_base_across_tags_wo_longevity.csv", from_date="2010-01-01", to_date="2025-12-31",max_pages = 45, page_size=100, ageing_tag_in = False, per_key=False)

QUERY: "healthy aging" OR "healthy ageing" OR "anti-ageing" OR "anti-aging" OR "anti ageing" OR "anti aging" OR "biohacking" OR "living longer" OR "well ageing" OR "well aging" OR "ageing well" OR "aging well" OR "slow ageing" OR "slow aging" OR "health rejuvenation" OR "nootropics"
['healthy aging', 'healthy ageing', 'anti-ageing', 'anti-aging', 'anti ageing', 'anti aging', 'biohacking', 'living longer', 'well ageing', 'well aging', 'ageing well', 'aging well', 'slow ageing', 'slow aging', 'health rejuvenation', 'nootropics']: Total matching articles found = 1,651


Guardian pages: 100%|███████████████████████████| 16/16 [00:58<00:00,  3.69s/it]


Saved 1650 rows to ../GuardianAPI/guardian_base_across_tags_wo_longevity.csv


In [ ]:
for year in range(2010,2025+1):
    from_date = f"{year}-01-01"
    to_date = f"{year}-12-31"
    search_topic_by_keys(keys=None, filename="guardian_base_across_tags.csv", from_date="2010-01-01", to_date="2025-12-31",max_pages = 1, page_size=100, ageing_tag_in = False, per_key=False)

In [1]:
import feedparser
import pandas as pd

URL = "https://feeds.bbci.co.uk/news/health/rss.xml"

feed = feedparser.parse(URL)
items = feed["entries"]

df_bbc_health = pd.DataFrame([{
    "title": item.get("title"),
    "link": item.get("link"),
    "published": item.get("published"),
    "summary": item.get("summary"),
} for item in items])

print(df_bbc_health.head())


                                               title  \
0  Baby food firms told to make products healthie...   
1  Bereaved families feel 'ignored' over maternit...   
2     Hospital pays out after bacteria-linked deaths   
3  Pine nuts and goat's milk should get allergy l...   
4  More weight-loss drugs could be offered as par...   

                                                link  \
0  https://www.bbc.com/news/articles/cvgpld8p9rqo...   
1  https://www.bbc.com/news/articles/c1ejq4yy501o...   
2  https://www.bbc.com/news/articles/cm21xrm5dyzo...   
3  https://www.bbc.com/news/articles/c5ypgrzxx9go...   
4  https://www.bbc.com/news/articles/cgkr2g3e5p1o...   

                       published  \
0  Thu, 21 Aug 2025 23:42:39 GMT   
1  Thu, 21 Aug 2025 23:57:08 GMT   
2  Wed, 20 Aug 2025 13:19:32 GMT   
3  Wed, 20 Aug 2025 11:58:27 GMT   
4  Wed, 20 Aug 2025 00:51:07 GMT   

                                             summary  
0  Manufacturers who do not make products healthi...  